# 10 — Data Drift Detection & Retraining Pipeline

The churn model in notebook 04 was trained once, on one snapshot of data. In a real
deployment, that's the beginning of the model's life, not the end of it — the actual
customer population keeps changing after the model ships (new customers skew toward
different plans, a pricing change shifts the MonthlyCharges distribution, a product
change alters which add-ons are common), and none of that raises an error. The model
just keeps making predictions, quietly less trustworthy over time.

This notebook demonstrates the monitoring layer built in `src/drift.py`, checking two
different kinds of drift that catch different failure modes:

1. **Feature drift** — are today's inputs shaped differently than the inputs the model
   was trained on? Measured via **Population Stability Index (PSI)**, the standard
   metric for this in credit-risk and churn-model monitoring specifically.
2. **Performance drift** — even if inputs look the same, has the actual relationship
   between inputs and outcomes changed? This can only be checked once new data has
   known outcomes, so it necessarily lags feature-drift monitoring.

To demonstrate this honestly rather than just running it against unchanged data (where
nothing interesting would happen), this notebook deliberately simulates a real-world
shift — a price increase applied to a later slice of customers — the same way the
drift module's own test suite validates that PSI actually catches a shift rather than
just computing a number that always looks fine.


In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
from src.preprocessing import load_raw, preprocess, get_model_features
from src.model import train_xgboost
from src.drift import compute_feature_drift, compute_performance_drift, should_retrain, retrain_and_compare
from sklearn.metrics import roc_auc_score

customers = load_raw()
print(f'{len(customers):,} customers loaded')


7,043 customers loaded


## Establishing the training baseline

The first 70% of customers (by row order, standing in for "earlier" data) become the
training baseline. The remaining 30% simulate newly-arrived data — with a deliberate
price increase applied, to demonstrate the monitor actually catching a real shift
rather than reporting "all clear" against unchanged data, which wouldn't prove
anything about whether the detector works.


In [2]:
split_point = int(len(customers) * 0.7)
baseline = customers.iloc[:split_point].copy()
new_arrivals = customers.iloc[split_point:].copy()

# Simulate a real-world shift: a price increase on newly-acquired customers
new_arrivals = new_arrivals.copy()
new_arrivals['MonthlyCharges'] = new_arrivals['MonthlyCharges'] * 1.25

print(f'Baseline: {len(baseline):,} customers')
print(f'New arrivals (with simulated price increase): {len(new_arrivals):,} customers')
print(f'Baseline avg MonthlyCharges: ${baseline["MonthlyCharges"].mean():.2f}')
print(f'New arrivals avg MonthlyCharges: ${new_arrivals["MonthlyCharges"].mean():.2f}')


Baseline: 4,930 customers
New arrivals (with simulated price increase): 2,113 customers
Baseline avg MonthlyCharges: $66.58
New arrivals avg MonthlyCharges: $84.31


## Checking feature drift

PSI is computed per feature, comparing the new arrivals' distribution against the
baseline's. Values below 0.10 are considered stable; 0.10-0.25 is a "watch" zone;
above 0.25 is a significant shift worth investigating (these bands are the same
industry-standard thresholds used in credit scorecard monitoring).


In [3]:
drift_report = compute_feature_drift(
    baseline, new_arrivals,
    features=['MonthlyCharges', 'tenure', 'Contract', 'InternetService', 'SeniorCitizen']
)
drift_report


,feature,psi,flag
0,MonthlyCharges,1.155646,alert
1,Contract,0.001875,stable
2,tenure,0.001284,stable
3,InternetService,0.000896,stable
4,SeniorCitizen,0.000000,stable


## The retrain decision

`should_retrain()` combines the feature-drift check with a performance-drift check
(if labeled new data is available) into a single decision, with the *reason* logged
explicitly — a retraining pipeline that fires without a stated reason is exactly the
kind of unexplainable behavior this project has avoided everywhere else (see the
"every decision is logged with its reasoning chain" principle in the Vertex Trading
onboarding docs this project's author has also worked with — the same discipline
applies here).


In [4]:
decision = should_retrain(drift_report, performance_drift=None)
for k, v in decision.items():
    print(f'{k}: {v}')


should_retrain: True
reasons: ["1 feature(s) with PSI > 0.25 (significant shift): ['MonthlyCharges']"]
n_alert_features: 1
n_watch_features: 0
checked_at: 2026-08-21T14:53:29.890667+00:00


## Performance drift: does the model's actual accuracy hold up?

Feature drift tells you inputs look different. It doesn't by itself prove the model
is worse — that requires checking the model's actual predictive performance against
new, labeled data. Here we simulate that by using the new arrivals' *actual* known
churn outcomes (available in this dataset, standing in for "enough time has now
passed to observe real outcomes" in a production setting) and scoring the
originally-trained model against them.


In [5]:
# Train the baseline model on the original 70% split
df_baseline = preprocess()  # full pipeline needed for feature engineering
df_baseline_slice = df_baseline.iloc[:split_point]
X_base, y_base = get_model_features(df_baseline_slice)
baseline_model = train_xgboost(X_base, y_base)
baseline_auc = roc_auc_score(y_base, baseline_model.predict_proba(X_base)[:, 1])
print(f'Baseline (training-time) AUC: {baseline_auc:.4f}')

# Score against the "new arrivals" slice (which include the simulated price shift)
df_new_slice = df_baseline.iloc[split_point:].copy()
df_new_slice['MonthlyCharges'] = df_new_slice['MonthlyCharges'] * 1.25  # apply the same shift used above
X_new, y_new = get_model_features(df_new_slice)

perf_drift = compute_performance_drift(baseline_auc, baseline_model, X_new, y_new)
for k, v in perf_drift.items():
    print(f'{k}: {v}')


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:53:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Baseline (training-time) AUC: 0.8741
baseline_auc: 0.874115941870848
new_data_auc: 0.7733541440902303
auc_drop: 0.10076179778061767
flag: alert


## Full retrain decision with both checks

Combining feature drift and performance drift into a single decision — this is what a
real monitoring job would evaluate on a schedule.


In [6]:
full_decision = should_retrain(drift_report, perf_drift)
for k, v in full_decision.items():
    print(f'{k}: {v}')


should_retrain: True
reasons: ["1 feature(s) with PSI > 0.25 (significant shift): ['MonthlyCharges']", 'AUC dropped 0.101 on new labeled data (0.874 -> 0.773)']
n_alert_features: 1
n_watch_features: 0
checked_at: 2026-08-21T14:53:30.405032+00:00


## Retraining and comparing

If the decision is to retrain, `retrain_and_compare()` trains a fresh model on the
updated data and reports a clear before/after comparison — a retrain that makes
things *worse* should be visible, not silently treated as an improvement just because
retraining happened.


In [7]:
if full_decision['should_retrain']:
    X_full_new = pd.concat([X_base, X_new])
    y_full_new = pd.concat([y_base, y_new])

    comparison = retrain_and_compare(
        train_fn=train_xgboost,
        X_train_new=X_full_new, y_train_new=y_full_new,
        X_test=X_new, y_test=y_new,
        old_model=baseline_model,
    )
    print(f"Old model AUC on new data: {comparison['old_model_auc']:.4f}")
    print(f"Retrained model AUC on new data: {comparison['new_model_auc']:.4f}")
    print(f"Improved: {comparison['improved']} (delta: {comparison['delta']:+.4f})")
else:
    print('No retrain triggered.')


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:53:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Old model AUC on new data: 0.7734
Retrained model AUC on new data: 0.8673
Improved: True (delta: +0.0939)


## Takeaway

The monitor correctly flagged the simulated price shift as a significant feature
drift event (MonthlyCharges PSI well above the alert threshold), and retraining on
the combined dataset recovered — or improved — the model's performance on the shifted
population. The important design point isn't the specific numbers here, which depend
on the size of the simulated shift; it's that the pipeline makes drift **visible and
explainable** rather than letting a model degrade silently, and it only acts
(retrains) with a documented, auditable reason rather than on a fixed schedule
regardless of whether anything has actually changed.
